# Predicting Corporate Bankruptcy from Financial Statements
### An Imbalanced Classification Approach to Company Risk Screening

**Author:** Mauro Reverberi

**Program:** MSc AI, Udacity Institute of AI & Technology / Woolf

**Project:** Capstone Project 3, Machine Learning Foundations

**Task type:** supervised learning, binary classification

**Dataset:** Polish Companies Bankruptcy Data, UCI Machine Learning Repository (dataset 365),
https://archive.ics.uci.edu/dataset/365/polish+companies+bankruptcy+data , licence CC BY 4.0

In this notebook I train a model that predicts from one year of financial statement ratios whether a company will go bankrupt within the next three years. The input is a set of 64 financial ratios computed from published annual statements, the output is a bankruptcy risk score.

I picked this problem because it continues my capstone series. In Project 1 I built a cleaned dataset of Swiss legal entities from the GLEIF register, in Project 2 I analyzed company mutations from the Swiss commercial gazette. A later capstone project will build a due diligence agent that looks up a company and assesses it. Such an agent needs exactly the decision modeled here, given the numbers a company publishes, how urgently does a human analyst need to look at it.

**Research question:** How well can supervised machine learning predict, from one year of financial ratios, whether a company will go bankrupt within three years, and how should the model scores be turned into decisions when bankruptcies are rare?

## Problem Definition and Dataset

**Task type: supervised learning, binary classification.** Given the financial ratios of one company in one year, the model predicts whether the company went bankrupt within the following three years.

The data was collected from the Emerging Markets Information Service (EMIS) and covers Polish companies, bankrupt ones observed in 2000 to 2012, still operating ones evaluated from 2007 to 2013. It was donated to the UCI Machine Learning Repository by Sebastian Tomczak and is described in Zieba, Tomczak and Tomczak (2016). Every label is a real observed company outcome, so the data is not synthetic. The archive contains five files, one per forecasting horizon, from 1year (ratios from the first year, bankruptcy label after five years) to 5year (ratios from the fifth year, label after one year). I use the 3year file and leave the other four untouched, it is the largest of the five and its three-year horizon fits the due diligence use case, a risk screening wants warning ahead of time, not a confirmation shortly before the collapse. The bankruptcy share of under 5% makes this an imbalanced classification problem, which shapes the metric choice and the threshold analysis below.

I downloaded the archive on 2026-08-24 and keep it unchanged in the project folder. The notebook reads the file it needs directly from the archive, so no manual unpacking step is required.

## Setup

In [1]:
import io
import textwrap
import time
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score,
                             confusion_matrix, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")

# set constants
ARCHIVE = "polish+companies+bankruptcy+data.zip"
YEARS = 3

## Load the dataset

I read the 3year file directly from the downloaded archive and check that the size and the class counts match the published description.

In [2]:
def load_dataset(archive_path, years):
    """Load one horizon file of the UCI Polish bankruptcy archive.

    The archive contains five ARFF files, one per forecasting horizon,
    and years selects the file (1 to 5). An ARFF file is a CSV table
    with a small header of @attribute lines. The "?" placeholder
    becomes a proper missing value and the target column "bankrupt"
    comes back as integers, 1 means bankrupt.
    """
    with zipfile.ZipFile(archive_path) as archive:
        with archive.open(f"{years}year.arff") as file:
            text = io.TextIOWrapper(file, encoding="utf-8").read()
    n_attributes = text.lower().count("@attribute") - 1
    columns = [f"Attr{i}" for i in range(1, n_attributes + 1)] + ["bankrupt"]
    data_section = text[text.lower().index("@data"):].split("\n", 1)[1]
    df = pd.read_csv(io.StringIO(data_section), header=None,
                     names=columns, na_values=["?"])
    df["bankrupt"] = df["bankrupt"].astype(int)
    return df

In [3]:
df = load_dataset(ARCHIVE, YEARS)

bankrupt_count = int(df["bankrupt"].sum())
print(f"Companies: {df.shape[0]:,}, columns: {df.shape[1]} "
      f"(64 financial ratios + target)")
print(f"Bankrupt within {YEARS} years: {bankrupt_count:,} "
      f"({df['bankrupt'].mean():.2%})")

Companies: 10,503, columns: 65 (64 financial ratios + target)
Bankrupt within 3 years: 495 (4.71%)
